# 05 - EKF on MuJoCo data: fixed gains (A) versus estimated gains (B)

1. Calibrate the odometry gains once, at a single operating point (as one would do in practice).
2. Drive the robot around the room at several speeds and log encoders, camera marker poses and ground truth. The waypoint follower reads the true pose; it only generates motion for the log and is not the project's controller.
3. Replay the log through the EKF variants. The filters never see the truth; it is used only to evaluate them.

Unlike the synthetic study, the real odometry gains here depend on speed, and the marker pose noise comes from rendered images.

In [ ]:
!pip install -q mujoco opencv-contrib-python
!test -d robot || git clone -q https://github.com/hongjiezheng-1-hue/robot.git
!cd robot && git pull -q
!test -d robotis_mujoco_menagerie || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ROBOTIS-GIT/robotis_mujoco_menagerie.git
!cd robotis_mujoco_menagerie && git sparse-checkout set robotis_tb3

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
import sys, time
sys.path.insert(0, "robot")
sys.path.insert(0, "robot/experiments")
import numpy as np
import matplotlib.pyplot as plt
from sim.scene import SceneConfig, write_scene
from sim.mujoco_run import RunConfig, collect_run, calibrate_gains, min_clearance
from drivers.mujoco_tb3 import MujocoTB3Driver
from replay import replay, metrics, print_table, odometry_ratio_by_speed, marker_residual_by_distance, CHI2_3_999

scene_cfg, run_cfg = SceneConfig(), RunConfig()
scene_path = write_scene(scene_cfg, "robotis_mujoco_menagerie/robotis_tb3", flip_marker=False)
drv = MujocoTB3Driver(scene_path)
print("driver ready, control period", drv.control_dt, "s")

In [ ]:
s_v_cal, s_w_cal = calibrate_gains(drv, run_cfg)
print(f"calibrated gains (one straight run at wheel speed 3 rad/s, one spin at 1 rad/s): s_v = {s_v_cal:.3f}, s_w = {s_w_cal:.3f}")
print(f"equivalent effective track width: {run_cfg.track_nominal / s_w_cal:.3f} m (nominal {run_cfg.track_nominal} m)")

In [ ]:
t0 = time.time()
run = collect_run(drv, scene_cfg, run_cfg)
np.savez("run_mujoco.npz", **run)
visible = np.mean(np.all(np.isfinite(run["marker"]), axis=1))
print(f"collected {len(run['truth'])} samples in {time.time() - t0:.0f} s; marker pose available in {100 * visible:.0f}% of samples")
print(f"smallest clearance of the robot centre to obstacles, station and walls: {min_clearance(run['truth'], scene_cfg):.2f} m (robot radius about 0.15 m)")

In [ ]:
ratios = odometry_ratio_by_speed(run)
print("true / nominal-odometry linear speed, near-straight intervals")
for lo, hi, n, r in ratios["v"]:
    print(f"   nominal speed {lo:.2f} to {hi:.2f} m/s: median ratio {r:.3f} (n={n})")
print("true / nominal-odometry yaw rate")
for lo, hi, n, r in ratios["w"]:
    print(f"   |nominal yaw rate| {lo:.2f} to {hi:.2f} rad/s: median ratio {r:.3f} (n={n})")

print("marker pose noise (measured minus true) by distance, against the filter's model")
print("   distance [m] |   n | x std [mm] | y std [mm] | heading std [deg] | model pos sigma [mm] | model heading sigma [deg]")
for lo, hi, n, sx, sy, sa, mp, ma in marker_residual_by_distance(run):
    print(f"   {lo:.1f} - {hi:.1f}   | {n:3d} | {1000 * sx:10.1f} | {1000 * sy:10.1f} | {sa:17.2f} | {1000 * mp:20.1f} | {ma:24.2f}")

In [ ]:
variants = [
    ("A0 nominal gains", dict(gains=(1.0, 1.0))),
    ("A calibrated", dict(gains=(s_v_cal, s_w_cal))),
    ("A calibrated + gate", dict(gains=(s_v_cal, s_w_cal), gate=CHI2_3_999)),
    ("B estimated gains", dict(estimate=True)),
    ("B + gate", dict(estimate=True, gate=CHI2_3_999)),
]
rows, results = [], {}
for name, kw in variants:
    errs, nees, hist, rejected = replay(run, **kw)
    rows.append((name, metrics(errs, nees), rejected))
    results[name] = (errs, nees, hist)
print_table(rows)
print(f"B final gains: s_v = {results['B estimated gains'][2][-1, 0]:.3f}, s_w = {results['B estimated gains'][2][-1, 1]:.3f}")

In [ ]:
t = np.arange(1, len(run["truth"]) + 1) * run["dt"]
fig, axs = plt.subplots(3, 1, figsize=(10, 11))
for name, (errs, nees, hist) in results.items():
    axs[0].plot(t, 1000 * np.linalg.norm(errs[:, :2], axis=1), lw=0.9, label=name)
axs[0].set_yscale("log"); axs[0].set_xlabel("time [s]"); axs[0].set_ylabel("position error [mm]")
axs[0].set_title("EKF position error on the MuJoCo run"); axs[0].legend(fontsize=8); axs[0].grid(True, which="both", alpha=0.3)
hb = results["B estimated gains"][2]
axs[1].plot(t, hb[:, 0], label="B s_v"); axs[1].plot(t, hb[:, 1], label="B s_w")
axs[1].axhline(s_v_cal, ls="--", c="C0", lw=0.8, label="calibrated s_v"); axs[1].axhline(s_w_cal, ls="--", c="C1", lw=0.8, label="calibrated s_w")
axs[1].set_xlabel("time [s]"); axs[1].set_ylabel("odometry gain"); axs[1].legend(fontsize=8); axs[1].grid(True, alpha=0.3)
axs[2].plot(run["truth"][:, 0], run["truth"][:, 1], lw=0.7, label="true path")
seen = np.all(np.isfinite(run["marker"]), axis=1)
axs[2].plot(run["truth"][seen, 0], run["truth"][seen, 1], ".", ms=2, c="C3", label="marker detected")
axs[2].plot(*run["marker_world"][:2], "gs", label="charging station")
axs[2].set_aspect("equal"); axs[2].legend(fontsize=8); axs[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("ekf_mujoco_results.png", dpi=150); plt.show()

## What to look for

- The speed table shows whether the true-to-nominal ratio changes with speed (the synthetic study assumed it did not).
- The marker noise table shows whether the filter's measurement-noise model matches the rendered measurements.
- Compare A calibrated, B and the gated variants on position error and ANEES (ideal 3).